In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.630800                    0.457200   
             precision                   0.546184                    0.380803   
             recall                      0.640039                    0.556666   
             f1                          0.588329                    0.451355   
             kappa                       0.257855                   -0.048695   
             MCC                         0.261319                   -0.052644   
outputsTest  accuracy                    0.633600                    0.448000   
             precision                   0.547261                    0.368034   
             recall                      0.653914                    0.540227   
             f1                          0.594967                    0.436940   
             kappa                       0.265483                   -0.068066   
             MCC                         0.269560                   -0.072603   
outputsAll   accuracy                    0.630200                    0.457300   
             precision                   0.546452                    0.379234   
             recall                      0.640249                    0.551403   
             f1                          0.588704                    0.448452   
             kappa                       0.256699                   -0.050640   
             MCC                         0.259897                   -0.054646   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.591300   
             precision                         0.661719   
             recall                            0.601744   
             f1                                0.629456   
             kappa                             0.175716   
             MCC                               0.177278   
outputsTest  accuracy                          0.589400   
             precision                         0.661205   
             recall                            0.597970   
             f1                                0.626986   
             kappa                             0.172121   
             MCC                               0.174021   
outputsAll   accuracy                          0.596700   
             precision                         0.667475   
             recall                            0.604079   
             f1                                0.633160   
             kappa                             0.187922   
             MCC                               0.189826   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.557200   
             precision                      0.408142   
             recall                         0.572380   
             f1                             0.475066   
             kappa                          0.111179   
             MCC                            0.116509   
outputsTest  accuracy                       0.555600   
             precision                      0.410615   
             recall                         0.574947   
             f1                             0.477826   
             kappa                          0.109259   
             MCC                            0.114763   
outputsAll   accuracy                       0.552300   
             precision                      0.399413   
             recall                         0.561997   
             f1                             0.465471   
             kappa                          0.099779   
             MCC                            0.104812   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.553600                  0.476700  
             precision                  0.262234                  0.151407  
             recall                     0.590573                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.263592,-0.059965,0.180375,0.112028,0.108606,-0.082481
accuracy,0.631533,0.454167,0.592467,0.555033,0.552967,0.480567
f1,0.590667,0.445582,0.629867,0.472788,0.361769,0.219548
kappa,0.260012,-0.055800,0.178586,0.106739,0.090958,-0.065614
precision,0.546632,0.376024,0.663466,0.406057,0.262673,0.153371
recall,0.644734,0.549432,0.601264,0.569775,0.589042,0.393429


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.284576                    0.003401   
             spearman                   0.276861                   -0.009886   
             MSE                        1.430848                    1.993198   
             RMSE                       1.193438                    1.410182   
             MAE                        0.943995                    1.106920   
outputsTest  pearson                    0.271097                   -0.020338   
             spearman                   0.263807                   -0.034493   
             MSE                        1.457806                    2.040676   
             RMSE                       1.205342                    1.426401   
             MAE                        0.950079                    1.121878   
outputsAll   pearson                    0.278243                   -0.014168   
             spearman                   0.267381                   -0.020610   
             MSE                        1.443514                    2.028337   
             RMSE                       1.199567                    1.422551   
             MAE                        0.944601                    1.115464   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.149631   
             spearman                         0.207040   
             MSE                              1.700738   
             RMSE                             1.301885   
             MAE                              0.999198   
outputsTest  pearson                          0.138545   
             spearman                         0.207491   
             MSE                              1.722910   
             RMSE                             1.310885   
             MAE                              1.005133   
outputsAll   pearson                          0.154995   
             spearman                         0.217229   
             MSE                              1.690011   
             RMSE                             1.297965   
             MAE                              0.992457   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.119450                  0.064364   
             spearman                      0.135546                  0.143936   
             MSE                           1.761101                  1.871271   
             RMSE                          1.325150                  1.365560   
             MAE                           0.990625                  1.002925   
outputsTest  pearson                       0.125782                  0.061469   
             spearman                      0.134849                  0.143763   
             MSE                           1.748437                  1.877061   
             RMSE                          1.320109                  1.368003   
             MAE                           0.994108                  1.002279   
outputsAll   pearson                       0.126141                  0.061778   
             spearman                      0.121953                  0.136483   
             MSE                           1.747718                  1.876444   
             RMSE                          1.320369                  1.367589   
             MAE                           0.995042                  1.005176   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.038134  
             spearman                 -0.082911  
             MSE                       2.076268  
             RMSE                      1.439109  
             MAE                       1.049059  
outputsTest  pearson                  -0.008748  
             spearman                 -0.069977  
             MSE                       2.017495  
             RMSE                      1.418761  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.946225,1.114754,0.998929,0.993258,1.003460,1.046071
MSE,1.444056,2.020737,1.704553,1.752419,1.874925,2.049489
RMSE,1.199449,1.419711,1.303579,1.321876,1.367051,1.429852
pearson,0.277972,-0.010368,0.147724,0.123791,0.062537,-0.024744
spearman,0.269350,-0.021663,0.210587,0.130782,0.141394,-0.077226
